# Notebook 01: 理论推导与环境搭建

## 高斯波包入射势垒的含时量子力学模拟

---

## 1. 物理问题

研究高斯波包（量子粒子的波函数描述）入射到不同势垒时的含时演化过程，观察**量子隧穿**、**部分反射/透射**、**共振隧穿**等量子力学现象。

采用**自然单位**：$\hbar = m = 1$，使公式简洁、物理直觉清晰。

## 2. 含时薛定谔方程 (TDSE)

一维含时薛定谔方程：

$$i\hbar\frac{\partial\psi}{\partial t} = \hat{H}\psi = \left[-\frac{\hbar^2}{2m}\frac{\partial^2}{\partial x^2} + V(x)\right]\psi(x,t)$$

自然单位下简化为：

$$i\frac{\partial\psi}{\partial t} = \left[-\frac{1}{2}\frac{\partial^2}{\partial x^2} + V(x)\right]\psi(x,t)$$

形式解：

$$\psi(x, t+\Delta t) = e^{-i\hat{H}\Delta t}\psi(x,t)$$

## 3. 高斯波包初态

初始波函数取为高斯波包：

$$\psi(x,0) = \left(\frac{1}{2\pi\sigma^2}\right)^{1/4} \exp\left[-\frac{(x-x_0)^2}{4\sigma^2}\right] \exp(ik_0 x)$$

参数含义：
- $x_0$：波包中心初始位置（位于势垒左侧）
- $\sigma$：波包空间宽度，$\Delta x = \sigma$
- $k_0$：中心波矢，对应初始动量 $p_0 = \hbar k_0 = k_0$
- 初始动能：$\langle E \rangle = \frac{k_0^2}{2}$

Heisenberg 不确定度关系：$\Delta x \cdot \Delta p = \sigma \cdot \frac{1}{2\sigma} = \frac{1}{2}$（最小不确定度波包）

## 4. 三种势垒

### 4.1 方势垒 (Square Barrier)

$$V(x) = \begin{cases} V_0, & a < x < b \\ 0, & \text{otherwise} \end{cases}$$

- 最简单的势垒模型
- 隧穿区域：$E < V_0$，透射系数指数衰减 $T \sim e^{-2\kappa L}$，其中 $\kappa = \sqrt{2(V_0-E)}$，$L = b-a$
- 经典透射：$E > V_0$，仍有部分反射（量子反射）
- 解析透射系数：
$$T = \frac{1}{1 + \frac{V_0^2\sinh^2(\kappa L)}{4E(V_0-E)}}$$

### 4.2 Eckart 势

$$V(x) = \frac{V_0}{\cosh^2[\alpha(x-x_c)]}$$

- 光滑势，不存在不连续点（数值更稳定）
- **存在解析透射系数**，可用于验证数值方法：

$$T = \frac{\sinh^2(\pi k / \alpha)}{\sinh^2(\pi k / \alpha) + \cosh^2(\pi s)}$$

其中 $k = \sqrt{2E}$，$s = \sqrt{2V_0/\alpha^2 - 1/4}$

### 4.3 双势垒 (Double Barrier)

$$V(x) = V_0, \quad x \in [a_1, b_1] \cup [a_2, b_2]$$

- 两方势垒之间形成量子阱
- 阱中存在准束缚态（共振态）
- 当入射能量与共振能量匹配时，透射率接近1（**共振隧穿**）
- 这是共振隧穿二极管 (RTD) 的物理基础

## 5. Split-Operator FFT 方法

### 5.1 算法原理

将哈密顿量分为动能和势能两部分：$\hat{H} = \hat{T} + \hat{V}$

利用 **Trotter-Suzuki 分解**（二阶精度）：

$$e^{-i\hat{H}\Delta t} \approx e^{-i\hat{V}\Delta t/2} \cdot e^{-i\hat{T}\Delta t} \cdot e^{-i\hat{V}\Delta t/2} + \mathcal{O}(\Delta t^3)$$

### 5.2 计算步骤

1. **势能半步**（坐标空间）：$\psi \leftarrow e^{-iV(x)\Delta t/2}\,\psi$
2. **动能整步**（动量空间）：
   - $\tilde\psi = \mathcal{F}[\psi]$
   - $\tilde\psi \leftarrow e^{-ik^2\Delta t/2}\,\tilde\psi$
   - $\psi = \mathcal{F}^{-1}[\tilde\psi]$
3. **势能半步**（坐标空间）：$\psi \leftarrow e^{-iV(x)\Delta t/2}\,\psi$

### 5.3 优势

- 每步只需两次 FFT，$\mathcal{O}(N\log N)$
- 蛙跳格式，$\mathcal{O}(\Delta t^3)$ 误差
- **辛结构保持**：长时间概率守恒良好
- 无需构造矩阵，内存 $\mathcal{O}(N)$

## 6. 吸收边界条件 (CAP)

有限空间网格边界会反射波包，需添加**复吸收势**：

$$V_{\text{CAP}}(x) = -i\eta\left(\frac{|x - x_{\text{edge}}|}{L_{\text{CAP}}}\right)^n, \quad n = 3$$

CAP 在边界区域逐渐消耗概率流，模拟无穷远边界条件。

- $L_{\text{CAP}}$：吸收区域宽度
- $\eta$：吸收强度
- $n$：阶数（越高，边界反射越小）

## 7. 空间网格与 Nyquist 条件

- 空间网格：$x \in [x_{\min}, x_{\max}]$，$N_x = 2^n$ 个点
- $\Delta x = (x_{\max} - x_{\min})/(N_x - 1)$
- 动量网格：$k \in [-k_{\max}, k_{\max}]$，$k_{\max} = \pi/\Delta x$
- **Nyquist 条件**：$k_0 < k_{\max}$，即 $\Delta x < \pi/k_0$
- 典型参数：$N_x = 2048$，$\Delta x \approx 0.078$，$k_{\max} \approx 40 \gg k_0$

## 8. 环境搭建

In [ ]:
import sys
sys.path.insert(0, '.')

import numpy as np
import matplotlib.pyplot as plt
from config import SimParams, tunneling, above_barrier, resonant
from potentials import square_barrier, eckart_barrier, double_barrier, get_potential
from wavepacket import gaussian_wavepacket
from propagator import propagate
from observables import compute_observables, eckart_analytical_T_simple
from visualization import set_style, plot_potential_and_wavepacket

set_style()
print('All modules loaded successfully!')

## 9. 参数检查

In [ ]:
p = SimParams()
print(f'Spatial grid: x in [{p.x_min}, {p.x_max}], N_x = {p.N_x}')
print(f'Δx = {p.dx:.4f}, k_max = {np.pi/p.dx:.2f}')
print(f'Momentum grid: k in [{p.k.min():.2f}, {p.k.max():.2f}]')
print(f'Time step: dt = {p.dt}, Total time = {p.t_total:.2f}')
print(f'Wavepacket: x0 = {p.x0}, sigma = {p.sigma}, k0 = {p.k0}')
print(f'Initial kinetic energy E = {p.E_kinetic:.2f}, Barrier height V0 = {p.V0}')
print(f'E < V0: Tunneling regime' if p.E_kinetic < p.V0 else 'E > V0: Classical transmission regime')

In [ ]:
x = p.x
psi = gaussian_wavepacket(x, p)
norm = np.trapezoid(np.abs(psi)**2, x)
print(f'Initial wavepacket normalization: int|psi|^2dx = {norm:.10f}')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, name in zip(axes, ['square', 'eckart', 'double']):
    V = np.real(get_potential(x, p, name, with_cap=False))
    rho = np.abs(psi)**2
    ax.plot(x, V, 'k-', linewidth=2, label='V(x)')
    ax.axhline(p.E_kinetic, color='r', linestyle='--', label=f'E = {p.E_kinetic:.2f}')
    ax.fill_between(x, 0, rho * p.V0 / rho.max() * 0.6, alpha=0.4, label='|psi|^2')
    ax.set_xlim(p.x_min + p.cap_width, p.x_max - p.cap_width)
    ax.set_xlabel('x')
    ax.set_title(name)
    ax.legend()

plt.suptitle('Three Barriers and Initial Wavepacket', fontsize=16)
plt.tight_layout()
plt.show()

## 10. 预设参数方案

| 方案 | V₀ | k₀ | E_kinetic | 物理情境 |
|------|----|----|-----------|----------|
| `tunneling()` | 15 | 5 | 12.5 | E < V₀, 隧穿区域 |
| `above_barrier()` | 5 | 5 | 12.5 | E > V₀, 经典透射+量子反射 |
| `resonant()` | 10 | 4 | 8.0 | 双势垒共振隧穿 |